# Nodesetの所属変更

---

Nodeset の Partition への追加所属、および Partition からの所属解除を行います。Nodeset 自体の作成・削除は行いません。

## 概要

このNotebookでは、既存のPartitionとNodesetの所属関係を変更します：

1. **追加所属**: 既存の Nodeset を別の Partition にも所属させる（新規 Partition の作成も可能）
2. **所属解除**: Partition から Nodeset の所属を外す

それぞれ独立したセクションになっています。必要なセクションのみ実行してください。

### 操作の制約

- **Nodeset の作成・削除はできません**: Feature（Nodeset）自体の作成は「083-Featureの追加.ipynb」、削除は「084-Featureの削除.ipynb」を使用してください
- **どの Partition にも属さない Nodeset は許容しません**: 所属解除により Nodeset が孤立する場合はエラーになります。Feature ごと削除する場合は 084 を使用してください
- **空の Partition は自動削除されます**: 所属解除により Partition の Nodeset が空になった場合、その Partition は自動的に削除されます

### 前提条件

- 初期構築（010, 020）が完了していること
- 変更対象の Feature と Partition が存在すること

> VCノードの操作がないため、VCCアクセストークンは不要です。

## 準備

### UnitGroup名の指定

構築環境の UnitGroup名を指定します。

VCノードを作成時に指定した値を確認するために `group_vars` ファイル名の一覧を表示します。

In [ ]:
!ls -1 group_vars/*.yml | sed -e 's/^group_vars\///' -e 's/\.yml//' | sort

UnitGroup名を次のセルに指定してください。

In [ ]:
# (例)
# ugroup_name = 'OpenHPC'

ugroup_name = 

#### チェック

指定されたUnitGroup名が適切であることを確認します。

group_vars にfeature, partition 設定が記録されていることを確認します。

In [ ]:
%run scripts/group.py
gvars = load_group_vars(ugroup_name)

# group_vars にslurm_features, slurm_partitions が定義されていることを確認する
if not {"slurm_features", "slurm_partitions"} <= set(gvars.keys()):
    raise RuntimeError("Feature, Partition設定が未実施です")

Ansibleで接続できることを確認します。

In [ ]:
!ansible {ugroup_name}_master -m ping

### 現在の構成を確認する

現在の Partition, Feature 構成を確認します。

まず現在の登録されているPartitionとそれに属するNodesetの構成を表示します。

In [ ]:
%run scripts/group.py

gvars = load_group_vars(ugroup_name)

print("=== Partition / Nodeset 構成 ===")
for pname, pconfig in gvars["slurm_partitions"].items():
    default_mark = " [DEFAULT]" if pconfig.get("default", False) else ""
    nodesets = pconfig["nodesets"]
    print(f"  - {pname}{default_mark}")
    print(f"      Nodesets: {', '.join(nodesets)}")

現在登録されているFeatureの一覧を表示します。

In [ ]:
print("\n=== Feature 一覧 ===")
for fname, cfg in gvars.get("slurm_features", {}).items():
    tp = cfg["hostname_template"]
    max_nodes = len(cfg["ip_addresses"])
    partitions = find_affected_partitions(gvars, fname)
    print(f"""  - {fname}
        Provider: {cfg['provider']}
        Hosts: {tp.format(1)} .. {tp.format(max_nodes)}
        Partitions: {', '.join(partitions)}""")

## Nodesetの追加所属（オプション）

既存の Nodeset を別の Partition にも所属させる場合はこのセクションを実行してください。
変更しない場合はこのセクションをスキップしてください。

追加所属により、同一の計算ノードグループに対して複数の Partition から異なるスケジューリングポリシーでジョブを投入できるようになります。

### 追加する Feature の指定

Partitionに追加所属させるFeature名を指定してください。

> 実際の追加処理ではFeatureから自動的に導出されるNodeset名をPartitionに追加する設定を行います。ここではFeature名を指定してください。

In [ ]:
# (例)
# add_nodeset_feature = 'gpu-a100-aws'

add_nodeset_feature = 

### 追加先の Partition の指定

Nodeset を追加する Partition 名を指定してください。既存の Partition 名を指定することも、新規の Partition 名を指定することも可能です。

新規 Partition を作成する場合は、デフォルトPartitionの指定は `False` で作成されます。また AllowGroups の設定が必要な場合は、このNotebook実行後に「085-Partitionの変更.ipynb」で設定してください。

In [ ]:
# (例)
# add_nodeset_partition = 'general'

add_nodeset_partition = 

## Nodesetの所属解除（オプション）

Partition から Nodeset の所属を外す場合はこのセクションを実行してください。
変更しない場合はこのセクションをスキップしてください。

所属解除により Nodeset がどの Partition にも属さなくなる場合はエラーになります。
その場合は「084-Featureの削除.ipynb」で Feature ごと削除してください。
また、所属解除により Partition の Nodeset が空になった場合、その Partition は自動的に削除されます。

### 解除する Nodeset の指定

所属を解除する Feature 名を指定してください。

In [ ]:
# (例)
# remove_nodeset_feature = 'gpu-a100-aws'

remove_nodeset_feature = 

### 解除元の Partition の指定

Nodeset を除外する Partition 名を指定してください。

In [ ]:
# (例)
# remove_nodeset_partition = 'general'

remove_nodeset_partition = 

## 変更内容の確認

これまでに指定した変更内容をまとめて表示します。変更がない場合はその旨が表示されます。

In [ ]:
%run scripts/group.py
gvars = load_group_vars(ugroup_name)
changes = []

# 追加所属のチェック
if "add_nodeset_feature" in vars() and add_nodeset_feature:
    is_new_partition = check_nodeset_addable_to_partition(gvars, add_nodeset_feature, add_nodeset_partition)
    ns = f"ns-{add_nodeset_feature}"
    if is_new_partition:
        changes.append(f"  追加所属: {ns} → 新規 Partition '{add_nodeset_partition}' を作成")
    else:
        changes.append(f"  追加所属: {ns} → 既存 Partition '{add_nodeset_partition}' に追加")

# 所属解除のチェック
if "remove_nodeset_feature" in vars() and remove_nodeset_feature:
    deleted_partitions = check_nodeset_removable_from_partition(
        gvars, remove_nodeset_feature, remove_nodeset_partition
    )
    ns = f"ns-{remove_nodeset_feature}"
    changes.append(f"  所属解除: {ns} ← Partition '{remove_nodeset_partition}'")
    if deleted_partitions:
        for dp in deleted_partitions:
            changes.append(f"  Partition削除: '{dp}'（Nodeset が空になるため）")

if changes:
    print("変更内容:")
    for c in changes:
        print(c)
else:
    print("変更はありません。")
    print("add_nodeset_feature または remove_nodeset_feature を指定してください。")

## 構成定義の更新

変更内容を `group_vars` に保存します。
この時点ではまだ Slurm クラスタには反映されていません。

In [ ]:
%run scripts/group.py

if not changes:
    print("変更がないためスキップします。")
else:
    # 追加所属
    if "add_nodeset_feature" in vars() and add_nodeset_feature:
        add_feature_to_partition(ugroup_name, add_nodeset_partition, add_nodeset_feature)
        if is_new_partition:
            print(f"✓ 新規 Partition '{add_nodeset_partition}' を作成し ns-{add_nodeset_feature} を追加しました")
        else:
            print(f"✓ ns-{add_nodeset_feature} を Partition '{add_nodeset_partition}' に追加しました")

    # 所属解除
    if "remove_nodeset_feature" in vars() and remove_nodeset_feature:
        remove_feature_from_partition(ugroup_name, remove_nodeset_partition, remove_nodeset_feature)
        print(f"✓ ns-{remove_nodeset_feature} を Partition '{remove_nodeset_partition}' から除外しました")
        if deleted_partitions:
            for dp in deleted_partitions:
                print(f"✓ Partition '{dp}' を削除しました（Nodeset が空になったため）")

`group_vars` ファイルの内容を確認します。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## Slurmへの反映

構成定義の変更をSlurmの設定ファイルに反映し、その再読み込みを行います。

ansible で `slurm.conf`を再生成し、Slurm に設定を再読み込みさせます。まずチェックモードで実行します。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v -CD playbooks/setup-slurm.yml

実際に変更を行います。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v playbooks/setup-slurm.yml

Slurm にPartition の設定が正しく反映されていることを確認します。

In [ ]:
print("=== Partition / Nodeset 一覧 ===")
!ansible {ugroup_name}_master -m shell \
    -a "grep -wE 'PartitionName|NodeSet' /etc/slurm/slurm.conf"

print("\n=== Slurmクラスタの状態 ===")
!ansible {ugroup_name}_master \
    -a "sinfo -o '%P %N %F %f'"